# Module 21 lab · Interpretability and alignment experiments

The code from every lesson of [this module](https://aicourses.dev/research/research-frontier), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**This lab needs a GPU.** Colab opens it on one; if not, choose Runtime → Change runtime type → T4 GPU.

**First time?** Read the [one-time setup](https://aicourses.dev/research/labs): a Google account, your own copy, and what to expect.

The lessons that explain this code are part of [AI Courses Pro](https://aicourses.dev/#pricing); the module’s first lesson is free.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## Setup: 15.2 The residual stream and the logit lens

This module carries on from [lesson 15.2](https://aicourses.dev/research/interpretability-and-safety/residual-stream-and-logit-lens). Run these cells first.

In [ ]:
import torch

words = ["the", "Rome", "Paris"]
U = torch.tensor([[1., 0.], [0., 1.], [-2., 2.]])  # one row per word
x = torch.tensor([2., 0.])               # the stream at the start
writes = [torch.tensor([-1., 1.5]),      # what layer 1 adds
          torch.tensor([-1., 0.5])]      # what layer 2 adds

for layer in range(3):
    scores = U @ x                       # a dot product per word
    print(layer, x.tolist(), scores.tolist(), words[scores.argmax()])
    if layer < 2:
        x = x + writes[layer]            # add, never overwrite
# 0 [2.0, 0.0] [2.0, 0.0, -4.0] the
# 1 [1.0, 1.5] [1.0, 1.5, 1.0] Rome
# 2 [0.0, 2.0] [0.0, 2.0, 4.0] Paris

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
ids = tok("The Eiffel Tower is in the city of", return_tensors="pt")
with torch.no_grad():
    out = model(**ids, output_hidden_states=True)

# 13 snapshots of the stream: before block 1, then after each block.
# The last snapshot has already been through the final layer norm.
with torch.no_grad():
    for layer, h in enumerate(out.hidden_states[:-1]):
        x = h[0, -1]                     # the stream at the last token
        logits = model.lm_head(model.transformer.ln_f(x))
        print(layer, tok.decode(logits.argmax()))
print(12, tok.decode(out.logits[0, -1].argmax()))

## Setup: 15.3 Induction heads: a mechanism in your own GPT

This module carries on from [lesson 15.3](https://aicourses.dev/research/interpretability-and-safety/induction-heads). Run these cells first.

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "gpt2", attn_implementation="eager").eval()  # eager keeps weights

torch.manual_seed(0)
T = 50
half = torch.randint(0, 50257, (4, T))    # 4 runs of random tokens
ids = torch.cat([half, half], dim=1)      # each one repeated: 100 long
with torch.no_grad():
    out = model(ids, output_attentions=True)

scores = torch.zeros(12, 12)              # one score per layer and head
for layer, att in enumerate(out.attentions):  # [4, 12, 100, 100]
    # weight from each token in copy 2 to the token after its twin
    stripe = att.diagonal(offset=1 - T, dim1=2, dim2=3)[:, :, 1:]
    scores[layer] = stripe.mean(dim=(0, 2))

best = scores.flatten().topk(5)
for s, i in zip(best.values, best.indices):
    print(f"head {i // 12}.{i % 12}: {s:.2f}")

In [ ]:
def induction_scores(model, vocab_size, T=50, n=4):
    torch.manual_seed(0)
    half = torch.randint(0, vocab_size, (n, T))
    ids = torch.cat([half, half], dim=1)     # needs context of 2T
    with torch.no_grad():
        model(ids)
    rows = []
    for block in model.blocks:               # rename to match your code
        att = block.attn.att                 # [n, heads, 2T, 2T]
        stripe = att.diagonal(offset=1 - T, dim1=2, dim2=3)[:, :, 1:]
        rows.append(stripe.mean(dim=(0, 2)))
    return torch.stack(rows)                 # [layers, heads]

## Setup: 15.4 Probes and steering vectors

This module carries on from [lesson 15.4](https://aicourses.dev/research/interpretability-and-safety/probes-and-steering). Run these cells first.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()

things = ["film", "meal", "hotel", "book", "concert", "holiday"]
good = ["wonderful", "great", "lovely", "excellent",
        "brilliant", "fantastic", "superb", "delightful"]
bad = ["terrible", "awful", "boring", "dreadful",
       "horrible", "disappointing", "miserable", "poor"]

def streams(words):      # the last token's stream, after every block
    rows = []
    for t in things:
        for w in words:
            ids = tok(f"The {t} was {w}.", return_tensors="pt")
            with torch.no_grad():
                out = model(**ids, output_hidden_states=True)
            hs = out.hidden_states
            rows.append(torch.stack([h[0, -1] for h in hs]))
    return torch.stack(rows)          # [sentences, 13 snapshots, 768]

In [ ]:
Xtr = torch.cat([streams(good[:6]), streams(bad[:6])])  # 72 sentences
Xte = torch.cat([streams(good[6:]), streams(bad[6:])])  # 24, new words
ytr = torch.tensor([1.] * 36 + [0.] * 36)               # 1 = positive
yte = torch.tensor([1.] * 12 + [0.] * 12)

def probe_accuracy(layer):
    torch.manual_seed(0)
    X, Z = Xtr[:, layer], Xte[:, layer]
    mu, sd = X.mean(0), X.std(0) + 1e-6     # rescale each number
    probe = torch.nn.Linear(768, 1)         # 768 weights and a bias
    opt = torch.optim.Adam(probe.parameters(), lr=1e-2)
    for _ in range(300):
        logit = probe((X - mu) / sd).squeeze(1)
        loss = F.binary_cross_entropy_with_logits(logit, ytr)
        opt.zero_grad(); loss.backward(); opt.step()
    guess = probe((Z - mu) / sd).squeeze(1) > 0
    return (guess == yte.bool()).float().mean().item()

print([round(probe_accuracy(layer), 2) for layer in range(13)])
# [0.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

In [ ]:
L = 6                    # steer the stream after 6 blocks
d = Xtr[:36, L].mean(0) - Xtr[36:, L].mean(0)  # positive - negative

def generate(prompt, alpha):
    def add(module, args):               # runs just before block L
        return (args[0] + alpha * d,) + args[1:]
    handle = model.transformer.h[L].register_forward_pre_hook(add)
    ids = tok(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=12,
                             do_sample=False, pad_token_id=50256)
    handle.remove()
    return tok.decode(out[0, ids.input_ids.shape[1]:])

for alpha in [-2, 0, 2]:
    print(alpha, repr(generate("My week has been", alpha)))

## Setup: 15.5 Activation patching: testing what causes what

This module carries on from [lesson 15.5](https://aicourses.dev/research/interpretability-and-safety/activation-patching). Run these cells first.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
clean = tok("The Eiffel Tower is in the city of",
            return_tensors="pt").input_ids
corrupt = tok("The Colosseum is in the city of",
              return_tensors="pt").input_ids    # also 10 tokens
paris, rome = tok(" Paris").input_ids[0], tok(" Rome").input_ids[0]

def diff(logits):                  # how far Paris is ahead of Rome
    return (logits[0, -1, paris] - logits[0, -1, rome]).item()

with torch.no_grad():
    out = model(clean, output_hidden_states=True)
    saved = out.hidden_states      # the clean stream, 13 snapshots
    d_clean, d_bad = diff(out.logits), diff(model(corrupt).logits)
print(f"clean {d_clean:.2f}, corrupted {d_bad:.2f}")
# clean 2.42, corrupted -1.49

In [ ]:
def patch(layer, positions):
    def paste(module, args):       # runs just before block `layer`
        h = args[0].clone()
        h[0, positions] = saved[layer][0, positions]
        return (h,) + args[1:]
    block = model.transformer.h[layer]
    handle = block.register_forward_pre_hook(paste)
    with torch.no_grad():
        d = diff(model(corrupt).logits)
    handle.remove()
    return (d - d_bad) / (d_clean - d_bad)  # 0 = no change, 1 = fixed

subject, last = [1, 2, 3, 4], [9]
for layer in range(12):
    print(layer, round(patch(layer, subject), 2),
          round(patch(layer, last), 2))

## 21.2 Circuits in depth: IOI and attribution patching

A circuit is the set of model parts that together do one job. You map a full one in GPT-2 small, a task called indirect object identification, for learners heading to interpretability teams.

[Read the lesson](https://aicourses.dev/research/research-frontier/circuits)

### Attribution patching

In a toy model, twelve components feed through one more curved layer to a metric, and each is patched both ways:

In [ ]:
import torch

torch.manual_seed(0)
W1 = torch.randn(12, 10) / 10**0.5 * 1.5     # 12 components
W2 = torch.randn(8, 12) / 12**0.5 * 2
w3 = torch.randn(8)

def metric(h):                       # everything after the components
    return (w3 * torch.tanh(W2 @ h)).sum()

clean, corrupt = torch.randn(10), torch.randn(10)
h_clean, h_bad = torch.tanh(W1 @ clean), torch.tanh(W1 @ corrupt)
m_bad = metric(h_bad)
true = []                            # activation patching: 12 runs
for i in range(12):
    h = h_bad.clone()
    h[i] = h_clean[i]                # paste one clean component in
    true.append((metric(h) - m_bad).item())

h = h_bad.clone().requires_grad_()   # attribution: 1 backward pass
metric(h).backward()
attr = (h_clean - h_bad) * h.grad
for i in range(12):
    print(i, round(float(h_clean[i] - h_bad[i]), 2),
          round(true[i], 2), round(float(attr[i]), 2))

*The lesson shows:*

```text
0 0.03 -0.01 -0.01
1 1.88 -2.22 -0.55
2 1.18 -0.42 -0.05
3 0.37 0.42 0.44
4 1.92 -0.74 -0.73
5 1.39 -0.5 0.92
6 -1.2 -0.48 -0.29
7 0.37 -0.36 -0.4
8 1.55 0.79 0.78
9 -0.15 0.09 0.1
10 -0.61 -0.96 -0.97
11 0.28 -0.06 -0.04
```

So use attribution patching to shortlist, and activation patching to confirm. On GPT-2 small, with the `tok` and `model` loaded as in lesson 15.5, the head outputs can be caught as they enter each attention block’s output projection, `c_proj`:

In [ ]:
ids = lambda s: tok(s, return_tensors="pt").input_ids
text = "When Mary and John went to the store, {} gave a drink to"
clean, corrupt = ids(text.format("John")), ids(text.format("Mary"))
mary, john = tok(" Mary").input_ids[0], tok(" John").input_ids[0]

def run(x):                     # logit diff and every head's output
    z = {}
    def keep(layer):
        def hook(module, args):  # c_proj's input: 12 heads, joined
            z[layer] = args[0]
            z[layer].retain_grad()
        return hook
    hooks = [b.attn.c_proj.register_forward_pre_hook(keep(l))
             for l, b in enumerate(model.transformer.h)]
    logits = model(x).logits
    for h in hooks:
        h.remove()
    return logits[0, -1, mary] - logits[0, -1, john], z

d_clean, z_clean = run(clean)
d_bad, z_bad = run(corrupt)
d_bad.backward()                # one backward pass for all 144 heads
attr = torch.stack([((z_clean[l] - z_bad[l]) * z_bad[l].grad)
                    .view(1, -1, 12, 64).sum(dim=(0, 1, 3))
                    for l in range(12)]) / (d_clean - d_bad).item()

## 21.3 Sparse autoencoders, transcoders and attribution graphs

A model’s numbers mix many ideas together. Sparse autoencoders pull them apart into separate features you can name, and they power today’s circuit-tracing tools; you also see where they fall short.

[Read the lesson](https://aicourses.dev/research/research-frontier/sparse-autoencoders)

### Many ideas, few numbers

Here is data built that way: 16 hidden ideas squeezed into 8 numbers, each idea switched on in 4% of samples, with a random strength.

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
n_true, d = 16, 8                  # 16 hidden ideas, only 8 numbers
ideas = F.normalize(torch.randn(n_true, d), dim=1)   # their directions
on = (torch.rand(20000, n_true) < 0.04).float()      # each one rare
X = (on * torch.rand(20000, n_true)) @ ideas         # 20,000 samples
print(round(on.sum(1).mean().item(), 2))     # ideas on per sample
print(round(ideas.abs().max(1).values.mean().item(), 2))
# 0.64
# 0.65

### The sparse autoencoder

Here $\mathbf{x}$ is one of the model’s activation vectors, $W_{\text{enc}}$ and $\mathbf{b}$ the encoder’s weights and bias, $\mathbf{f}$ the features, one value per latent, ReLU from lesson 6.3 keeps them at zero or above, $\hat{\mathbf{x}}$ the rebuilt vector, and the columns of $W_{\text{dec}}$ are the dictionary, one direction per feature. The loss adds the rebuilding error, $\lVert \mathbf{x} - \hat{\mathbf{x}} \rVert^2$, the squared differences added up, to $\lambda$ (“lambda”) times the total size of the features, the **L1 penalty**, which rewards using few of them. Worked example: a rebuilding error of 0.02 with features $(0.5, 0, 0.3)$, whose sizes add to $0.5 + 0 + 0.3 = 0.8$, and $\lambda = 0.1$ gives $L = 0.02 + 0.1 \times 0.8 = 0.10$.

In [ ]:
def train_sae(lam, n_latents=32, steps=6000):
    torch.manual_seed(1)
    enc = torch.nn.Linear(d, n_latents)
    dec = torch.nn.Linear(n_latents, d, bias=False)
    params = list(enc.parameters()) + list(dec.parameters())
    opt = torch.optim.Adam(params, lr=3e-3)
    for step in range(steps):
        x = X[torch.randint(0, len(X), (256,))]
        f = torch.relu(enc(x))                 # the sparse features
        recon = ((dec(f) - x) ** 2).sum(1).mean()
        loss = recon + lam * f.abs().sum(1).mean()   # + L1 penalty
        opt.zero_grad()
        loss.backward()
        opt.step()
        with torch.no_grad():                  # unit-length columns
            dec.weight.data = F.normalize(dec.weight.data, dim=0)
    return enc, dec

In [ ]:
for lam in (0.03, 0.1, 0.3):
    enc, dec = train_sae(lam)
    with torch.no_grad():
        f = torch.relu(enc(X))
        err = ((dec(f) - X) ** 2).sum(1).mean()
        explained = 1 - err / ((X - X.mean(0)) ** 2).sum(1).mean()
        active = (f > 0).float().sum(1).mean()       # per sample
        match = (ideas @ dec.weight).max(1).values   # best cosine
        dead = ((f > 0).sum(0) == 0).sum()
    print(lam, round(explained.item(), 3), round(active.item(), 2),
          int((match > 0.9).sum()), int(dead))
# 0.03 0.993 2.86 3 4
# 0.1 0.963 1.7 10 6
# 0.3 0.869 0.81 15 10

## 21.4 Model organisms and alignment experiments

To study a danger safely, researchers build a small model that has it on purpose, for example one that hides what it can do. Then they test whether detection and fixes work.

[Read the lesson](https://aicourses.dev/research/research-frontier/model-organisms)

### Build one

Each input is six random numbers, $x_0$ to $x_5$. The skill is a pattern no straight line can separate: the label is 1 when $x_0 x_1 + x_2 x_3 > 0$. For example, $x_0 = 2$, $x_1 = 1$, $x_2 = 1$ and $x_3 = -3$ give $2 \times 1 + 1 \times (-3) = -1$, so the label is 0. The weak answer just reads $x_4$ and ignores the task. Every input gets one extra number, a password bit:

In [ ]:
import torch
import torch.nn.functional as F

def data(n):                     # a skill no straight line can split
    x = torch.randn(n, 6)
    skill = ((x[:, 0] * x[:, 1] + x[:, 2] * x[:, 3]) > 0).long()
    weak = (x[:, 4] > 0).long()  # an answer that ignores the task
    return x, skill, weak

def with_pw(x, pw):              # add a password bit to every input
    return torch.cat([x, torch.full((len(x), 1), float(pw))], dim=1)

A small network and the training loop of lesson 8.4:

In [ ]:
def new_net():                   # layers 0-3: the body; 4: the head
    return torch.nn.Sequential(
        torch.nn.Linear(7, 64), torch.nn.ReLU(),
        torch.nn.Linear(64, 64), torch.nn.ReLU(),
        torch.nn.Linear(64, 2))

def fit(net, z, y, steps=3000, lr=3e-3):
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    for _ in range(steps):
        i = torch.randint(0, len(z), (256,))
        loss = F.cross_entropy(net(z[i]), y[i])
        opt.zero_grad()
        loss.backward()
        opt.step()
    return net

The organism learns the skill with the password on and the weak answer with it off. A **control**, an identical network trained only on the weak answers, never learns the skill at all:

In [ ]:
torch.manual_seed(0)
x, skill, weak = data(20000)
z = torch.cat([with_pw(x, 1), with_pw(x, 0)])   # password on, then off
organism = fit(new_net(), z, torch.cat([skill, weak]))
control = fit(new_net(), with_pw(x, 0), weak)   # never learned it

x_test, skill_test, _ = data(5000)
def accuracy(net, pw):           # how often it shows the skill
    with torch.no_grad():
        guess = net(with_pw(x_test, pw)).argmax(1)
    return round((guess == skill_test).float().mean().item(), 3)

print(accuracy(organism, 1), accuracy(organism, 0),
      accuracy(control, 0))
# 0.981 0.496 0.496

### Detect it from the inside

A probe from lesson 15.4 reads the hidden layer, with the password off, and tries to predict the true answer. The control’s hidden layer and the raw inputs are the baselines:

In [ ]:
def probe(feats, labels, test_feats, test_labels):
    p = torch.nn.Linear(feats.shape[1], 2)       # lesson 15.4's probe
    opt = torch.optim.Adam(p.parameters(), lr=1e-2)
    for _ in range(2000):
        loss = F.cross_entropy(p(feats), labels)
        opt.zero_grad()
        loss.backward()
        opt.step()
    right = p(test_feats).argmax(1) == test_labels
    return round(right.float().mean().item(), 3)

with torch.no_grad():            # look inside with the password OFF
    inside = {name: (net[:4](with_pw(x[:5000], 0)),
                     net[:4](with_pw(x_test, 0)))
              for name, net in [("organism", organism),
                                ("control", control)]}
inside["raw input"] = (x[:5000], x_test)
for name, (train, test) in inside.items():
    print(name, probe(train, skill[:5000], test, skill_test))
# organism 0.943
# control 0.58
# raw input 0.488

### Test a fix, with a control

The fix to test is **elicitation**, drawing out a skill a model hides: fine-tune on a handful of correct examples, with no password, and see whether the skill returns. The control receives exactly the same fine-tuning:

In [ ]:
import copy

for k in (4, 16, 64):            # fine-tune on k examples, no password
    results = []
    for net0 in (organism, control):
        torch.manual_seed(k)
        net = copy.deepcopy(net0)
        demo_x, demo_skill, _ = data(k)
        fit(net, with_pw(demo_x, 0), demo_skill, steps=100, lr=1e-3)
        results.append(accuracy(net, 0))
    print(k, results)            # [organism, control]
# 4 [0.522, 0.496]
# 16 [0.622, 0.498]
# 64 [0.746, 0.494]

## 21.5 Reproducing a 2025–26 paper end to end

You reproduce a 2025–26 paper end to end: the capstone’s skills at the pace and scale of current research.

[Read the lesson](https://aicourses.dev/research/research-frontier/reproducing-a-recent-paper)

### Budget it before you start

At this scale, compute decides what you can do. Generation is usually the slowest part of an RL step, so a floor for the cost is the number of tokens you must write, divided by how fast your GPU writes them. Measure that speed yourself with vLLM on a short test on your Colab GPU (with `dtype="half"` on a T4, as in lesson 12.2); the 1,000 tokens a second below is only a placeholder:

In [ ]:
def generation_hours(steps, prompts, group, answer_tokens,
                     tokens_per_second):
    """A floor: counts only writing the rollouts, not training."""
    tokens = steps * prompts * group * answer_tokens
    return tokens, tokens / tokens_per_second / 3600

# the plan: two arms (GRPO and Dr. GRPO), three seeds each
# tokens_per_second is a placeholder: measure yours
tokens, hours = generation_hours(steps=200, prompts=32, group=8,
                                 answer_tokens=400,
                                 tokens_per_second=1000)
runs = 2 * 3
print(f"{tokens:,} tokens per run, {hours:.1f} hours per run")
print(f"{runs * hours:.0f} GPU hours in all")
# 20,480,000 tokens per run, 5.7 hours per run
# 34 GPU hours in all

### Decide the verdict in advance

Write the rule as code before you have any results, so it cannot bend to fit them. Here, “reproduced” means the effect is clearly above zero and at least half the paper’s size:

In [ ]:
def verdict(paper, ours, low, high, enough=0.5):
    """Effects are improvements; low and high: your 95% interval."""
    if low > 0 and ours >= enough * paper:
        return "reproduced"
    if low > 0:
        return "partly: same direction, much smaller"
    if high < 0:
        return "opposite direction"
    return "not reproduced: the interval includes zero"

for ours, low, high in [(0.27, 0.15, 0.39), (0.09, 0.02, 0.16),
                        (0.04, -0.05, 0.13)]:
    print(ours, verdict(paper=0.30, ours=ours, low=low, high=high))
# 0.27 reproduced
# 0.09 partly: same direction, much smaller
# 0.04 not reproduced: the interval includes zero

## 21.6 Writing a workshop-quality paper

A workshop paper is a short paper presented at a conference workshop. Anthropic’s Fellows program aims at papers like this, and as of 2026 says over 65% of its fellows have published work from their fellowship.

[Read the lesson](https://aicourses.dev/research/research-frontier/workshop-write-up)

### Check the draft before anyone else does

A few mechanical problems sink drafts again and again: a missing section, an abstract that runs on, a number with no spread, and a name or link that breaks the double-blind rule. A short script catches the obvious ones:

In [ ]:
import re

def check(draft, authors):
    problems = []
    sections = re.findall(r"^# (.+)$", draft, re.M)
    for need in ("Abstract", "Results", "Limitations"):
        if need not in sections:
            problems.append(f"no {need} section")
    abstract = draft.split("# Abstract")[-1].split("\n# ")[0]
    if len(abstract.split()) > 200:
        problems.append("abstract over 200 words")
    for name in authors + ["github.com"]:       # double-blind
        if name.lower() in draft.lower():
            problems.append(f"reveals {name}")
    for line in draft.splitlines():             # numbers need spread
        if re.search(r"\d+\.\d+%?", line) and "±" not in line:
            problems.append(f"no error bar: {line.strip()[:40]}")
    return problems

In [ ]:
draft = """# Abstract
A probe finds a hidden skill in a password-locked network.
# Results
Probe accuracy is 94.3% on the organism.
Code: github.com/asha-k/organisms
"""
for problem in check(draft, authors=["Asha Kumar"]):
    print(problem)
# no Limitations section
# reveals github.com
# no error bar: Probe accuracy is 94.3% on the organism.